# Intent Router BERT 训练 Notebook

基于 **hfl/chinese-macbert-base** 训练一个双任务分类模型，替代现有的 LLM-based Intent Router。

## 任务定义

输入：说话人角色 + 发言文本（格式：`speaker: text`）

输出：
- `severity` — 严重程度：ignore / simple / complex
- `intent_type` — 意图类型：query_law / compute_compensation / draft_clause / summarize / record_only / strategy_advice / risk_evaluation / none

## 数据说明

训练数据来源于项目现有测试用例（7 个对话剧本 + 单元测试 + e2e 测试），并通过模板扩充至约 1000 条。

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 第一步：安装依赖
# ═══════════════════════════════════════════════════════════════

!pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn

# 验证安装
import transformers, torch, sklearn
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 第二步：导入库
# ═══════════════════════════════════════════════════════════════

import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertModel,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import os
import json

# 固定随机种子
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 设备
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 第三步：构建数据集

数据来源：
1. **现有测试用例**（~210 条）：从 7 个对话剧本、单元测试、e2e 测试中提取的真实标注样本
2. **模板扩充**（~790 条）：基于业务规则设计的模板，覆盖 6 大法律领域（劳动、合同、刑事、婚姻、房产、借贷、交通）

标注规则（与原 LLM Intent Router 一致）：
- **律师发言** → 默认 `ignore/none`，仅当含显式系统求助词（"系统/AI/帮我查"）时为 `simple/query_law`
- **客户转述**（以"说/他们说/公司认为"开头） → `simple/record_only`，不因法律名词升级
- **客户提问** → 按内容细分：法条查询/计算/策略/胜率/起草/总结
- **客户寒暄/应答** → `ignore/none`

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3.1 基础种子样本（来自现有测试集的真实标注）
# ═══════════════════════════════════════════════════════════════

SEED_SAMPLES = [
    # ====== 单元测试样本 ======
    ("client", "违法解除赔多少？", "simple", "query_law"),
    ("client", "律师你好", "ignore", "none"),
    ("client", "我该怎么跟公司谈？", "complex", "strategy_advice"),
    ("lawyer", "你签劳动合同了吗？", "ignore", "none"),
    ("lawyer", "根据第39条可以解除", "ignore", "none"),
    ("lawyer", "系统帮我查一下第47条", "simple", "query_law"),
    ("client", "说我不胜任工作。", "simple", "record_only"),
    ("client", "能赢吗？", "complex", "risk_evaluation"),

    # ====== e2e_role_aware_dialogue 样本（劳动纠纷 30 轮） ======
    ("lawyer", "你好，请坐。今天想咨询什么问题？", "ignore", "none"),
    ("client", "王律师您好，我被公司违法解除了。", "complex", "query_law"),
    ("lawyer", "您在公司工作多久了？", "ignore", "none"),
    ("client", "两年三个月。", "simple", "record_only"),
    ("lawyer", "月薪是多少，税前还是税后？", "ignore", "none"),
    ("client", "税前两万五。", "simple", "record_only"),
    ("lawyer", "解除通知是什么时候收到的？", "ignore", "none"),
    ("client", "5月1号口头通知的。", "simple", "record_only"),
    ("lawyer", "有书面解除通知吗？", "ignore", "none"),
    ("client", "还没有，只是主管口头说的。", "ignore", "none"),
    ("lawyer", "劳动合同签了吗，几年期？", "ignore", "none"),
    ("client", "签了，三年期的。", "simple", "record_only"),
    ("lawyer", "公司给出的解除理由是什么？", "ignore", "none"),
    ("client", "说我不胜任工作。", "simple", "record_only"),
    ("lawyer", "之前有没有绩效考核记录？", "ignore", "none"),
    ("client", "有的，但都是合格的。", "ignore", "none"),
    ("client", "我能拿多少赔偿？", "simple", "compute_compensation"),
    ("lawyer", "违法解除的话一般是2N。", "ignore", "none"),
    ("client", "N+1怎么算？", "simple", "compute_compensation"),
    ("lawyer", "N是工作年限，每满一年一个月工资。", "ignore", "none"),
    ("client", "那我该怎么跟公司谈？", "complex", "strategy_advice"),
    ("lawyer", "先准备证据清单。", "ignore", "none"),
    ("client", "需要准备哪些证据？", "complex", "query_law"),
    ("lawyer", "劳动合同、工资流水、解除通知、考勤记录。", "ignore", "none"),
    ("client", "竞业限制最长多久？", "simple", "query_law"),
    ("lawyer", "两年。", "ignore", "none"),
    ("client", "加班费按什么标准？", "simple", "query_law"),
    ("lawyer", "工作日1.5倍，周末2倍，法定节假日3倍。", "ignore", "none"),
    ("client", "能赢吗？", "complex", "risk_evaluation"),
    ("lawyer", "证据充分的话胜率很高，不用太担心。", "ignore", "none"),
    ("client", "谢谢王律师，我回去准备材料。", "ignore", "none"),

    # ====== 刑事辩护剧本（盗窃罪）样本 ======
    ("lawyer", "您好，请坐。是什么案件？", "ignore", "none"),
    ("client", "律师好，我老公被警察抓了，说是盗窃。", "simple", "record_only"),
    ("lawyer", "什么时候被抓的，现在关在哪里？", "ignore", "none"),
    ("client", "前天晚上在家被抓的，现在关在区看守所。", "simple", "record_only"),
    ("lawyer", "家属收到拘留通知书了吗？", "ignore", "none"),
    ("client", "收到了，写的是涉嫌盗窃罪，拘留期限 30 天。", "simple", "record_only"),
    ("lawyer", "他之前有没有案底？", "ignore", "none"),
    ("client", "有过，三年前因为盗窃判过半年，已经刑满释放了。", "simple", "record_only"),
    ("lawyer", "这次涉嫌盗窃了几次？", "ignore", "none"),
    ("client", "警察说查了监控，有 5 次，都是进超市偷东西。", "simple", "record_only"),
    ("lawyer", "偷的是什么东西，大概值多少钱？", "ignore", "none"),
    ("client", "都是烟酒和电子产品，具体金额我不清楚。", "simple", "record_only"),
    ("lawyer", "警方有没有告知涉案金额？", "ignore", "none"),
    ("client", "办案民警说初步认定 3 万多，但还没出正式鉴定。", "simple", "record_only"),
    ("lawyer", "5 次盗窃分别是什么时间？", "ignore", "none"),
    ("client", "从今年 1 月到 4 月，每个月一次，都是晚上去的。", "simple", "record_only"),
    ("lawyer", "他是怎么进超市的，撬锁还是翻墙？", "ignore", "none"),
    ("client", "他说后门没锁，推门进去的，没撬锁。", "simple", "record_only"),
    ("lawyer", "有没有同伙？", "ignore", "none"),
    ("client", "没有，他一个人干的。", "simple", "record_only"),
    ("lawyer", "赃物处理了吗，还在家里吗？", "ignore", "none"),
    ("client", "大部分卖了，还剩一些烟酒在家，警察已经搜走了。", "simple", "record_only"),
    ("lawyer", "销赃的钱还剩多少？", "ignore", "none"),
    ("client", "他说花了 1 万多，还剩几千块在银行卡里，已经被冻结了。", "simple", "record_only"),
    ("lawyer", "您老公现在的工作和家庭情况？", "ignore", "none"),
    ("client", "他之前在工地打工，我是超市收银员，有个孩子 5 岁。", "simple", "record_only"),
    ("lawyer", "家里经济条件怎么样？", "ignore", "none"),
    ("client", "一般，月收入加起来 8 千左右，没什么存款。", "simple", "record_only"),
    ("lawyer", "他为什么去偷，是欠了钱还是吸毒？", "ignore", "none"),
    ("client", "他说去年赌博欠了债，被催得紧，没办法才去偷的。", "simple", "record_only"),
    ("lawyer", "现在欠了多少赌债？", "ignore", "none"),
    ("client", "还有 5 万左右没还，高利贷。", "simple", "record_only"),
    ("client", "盗窃罪怎么量刑？", "simple", "query_law"),
    ("lawyer", "数额较大的三年以下，数额巨大的三到十年，数额特别巨大的十年以上。", "ignore", "none"),
    ("client", "3 万多算数额较大还是巨大？", "simple", "query_law"),
    ("lawyer", "各地标准不同，一般 3 万以上算数额巨大，基准刑三到十年。", "ignore", "none"),
    ("client", "累犯会加重吗？", "simple", "query_law"),
    ("lawyer", "累犯从重处罚，不能缓刑，一般加基准刑的 10% 到 40%。", "ignore", "none"),
    ("client", "多次盗窃会加重吗？", "simple", "query_law"),
    ("lawyer", "两年内三次以上盗窃就构成多次盗窃，属于入罪标准之一，也是从重情节。", "ignore", "none"),
    ("client", "能取保候审吗？", "complex", "query_law"),
    ("lawyer", "累犯取保比较难，但金额如果能有争议，或者有退赔情节，可以尝试。", "ignore", "none"),
    ("client", "我该请律师还是等法律援助？", "complex", "strategy_advice"),
    ("lawyer", "建议尽快请律师，侦查阶段只有律师能会见，能了解案情和警方证据。", "ignore", "none"),
    ("client", "律师会见能做什么？", "simple", "query_law"),
    ("lawyer", "了解具体案情、核实证据、提供法律咨询、告知权利义务、安抚情绪。", "ignore", "none"),
    ("client", "金额鉴定有问题怎么办？", "complex", "strategy_advice"),
    ("lawyer", "可以申请重新鉴定，特别是对电子产品的价格认定，常有争议空间。", "ignore", "none"),
    ("client", "退赃能轻判吗？", "simple", "query_law"),
    ("lawyer", "退赃是从轻情节，但累犯不能缓刑，减轻幅度有限。", "ignore", "none"),
    ("client", "认罪认罚能减多少？", "simple", "query_law"),
    ("lawyer", "认罪认罚一般减 20% 左右，但累犯也要从重，综合看最终刑期。", "ignore", "none"),
    ("client", "大概会判多久？", "complex", "risk_evaluation"),
    ("lawyer", "三到十年之间，具体看金额认定和从轻情节，大概四年左右。", "ignore", "none"),
    ("client", "能缓刑吗？", "complex", "risk_evaluation"),
    ("lawyer", "累犯不能缓刑，这是硬性规定。", "ignore", "none"),
    ("client", "谢谢律师，我们尽快筹钱请律师。", "ignore", "none"),

    # ====== 离婚财产剧本样本 ======
    ("lawyer", "您好，请坐。今天咨询什么问题？", "ignore", "none"),
    ("client", "律师好，我想离婚，财产分割比较复杂。", "simple", "record_only"),
    ("lawyer", "结婚多久了，有孩子吗？", "ignore", "none"),
    ("client", "结婚 8 年，有一个儿子 6 岁。", "simple", "record_only"),
    ("lawyer", "主要财产有哪些？", "ignore", "none"),
    ("client", "两套房子，一套婚前他买的，一套婚后共同买的。", "simple", "record_only"),
    ("lawyer", "婚后这套房子登记在谁名下？", "ignore", "none"),
    ("client", "登记在他一个人名下，但首付是我们一起出的。", "simple", "record_only"),
    ("lawyer", "还有其他财产吗？", "ignore", "none"),
    ("client", "他名下有一家公司，股权应该值不少钱。", "simple", "record_only"),
    ("lawyer", "公司是他婚前成立的还是婚后的？", "ignore", "none"),
    ("client", "婚后第三年成立的，注册资金 100 万，他占 80%。", "simple", "record_only"),
    ("lawyer", "公司盈利情况怎么样？", "ignore", "none"),
    ("client", "去年净利润大概 200 万，但他说公司亏钱。", "simple", "record_only"),
    ("lawyer", "您参与公司经营了吗？", "ignore", "none"),
    ("client", "没有，我一直在国企上班，没管过公司的事。", "simple", "record_only"),
    ("lawyer", "您自己的收入情况呢？", "ignore", "none"),
    ("client", "年薪 15 万左右，比较稳定。", "simple", "record_only"),
    ("lawyer", "他的收入呢？", "ignore", "none"),
    ("client", "工资卡上每月 2 万，但公司分红他不告诉我。", "simple", "record_only"),
    ("lawyer", "有股票、基金这些吗？", "ignore", "none"),
    ("client", "有一个股票账户，婚后开的，里面大概 30 万。", "simple", "record_only"),
    ("lawyer", "存款呢？", "ignore", "none"),
    ("client", "我知道的联名账户有 20 万，他个人账户我不清楚。", "simple", "record_only"),
    ("lawyer", "债务情况呢，有没有共同债务？", "ignore", "none"),
    ("client", "婚后房子还有 80 万贷款没还完，其他不知道。", "simple", "record_only"),
    ("client", "婚前那套房子我能分吗？", "simple", "query_law"),
    ("lawyer", "婚前财产原则上归个人，但婚后还贷部分及增值可以要求补偿。", "ignore", "none"),
    ("client", "公司股份我能分多少？", "simple", "query_law"),
    ("lawyer", "婚后取得的股权属于共同财产，原则上一人一半，但实务中可能折价补偿。", "ignore", "none"),
    ("client", "孩子抚养权一般怎么判？", "simple", "query_law"),
    ("lawyer", "6 岁孩子法院会综合考虑抚养能力、孩子意愿，您收入稳定是有利因素。", "ignore", "none"),
    ("client", "我怎么查清他到底有多少财产？", "complex", "strategy_advice"),
    ("lawyer", "可以申请法院调查令，查银行流水、股票账户、公司账册。", "ignore", "none"),
    ("client", "他转移财产怎么办？", "simple", "query_law"),
    ("lawyer", "可以申请财产保全，如果能证明他恶意转移，可以少分或不分给他。", "ignore", "none"),
    ("client", "如果走诉讼，我大概能拿到多少？", "complex", "risk_evaluation"),
    ("lawyer", "要看财产查清情况，粗略估计婚后共同财产部分您能分到 40% 到 60%。", "ignore", "none"),
    ("client", "抚养权我能拿到吗？", "complex", "risk_evaluation"),
    ("lawyer", "您收入稳定、孩子年龄小，胜率较高，但最终要看具体证据。", "ignore", "none"),
    ("client", "我该先协议还是直接起诉？", "complex", "strategy_advice"),
    ("lawyer", "建议先摸清财产底细再谈判，谈不拢再起诉，避免被动。", "ignore", "none"),
    ("client", "谈判的时候要注意什么？", "complex", "strategy_advice"),
    ("lawyer", "不要暴露底线，所有承诺要求书面确认，最好有律师在场。", "ignore", "none"),
    ("client", "谢谢律师，我回去整理一下材料。", "ignore", "none"),

    # ====== 房产纠纷剧本样本 ======
    ("lawyer", "您好，请问是什么房产纠纷？", "ignore", "none"),
    ("client", "律师好，我买的房子开发商逾期交房快一年了。", "simple", "record_only"),
    ("lawyer", "什么时候签的购房合同？", "ignore", "none"),
    ("client", "去年3月签的，合同约定今年3月交房。", "simple", "record_only"),
    ("lawyer", "逾期交房的违约金怎么约定的？", "ignore", "none"),
    ("client", "合同写每天万分之三，但开发商说资金困难不想赔。", "simple", "record_only"),
    ("lawyer", "房子现在是什么状态？", "ignore", "none"),
    ("client", "主体结构做完了，但绿化和装修还没开始。", "simple", "record_only"),
    ("lawyer", "您全款还是贷款？", "ignore", "none"),
    ("client", "首付30%，贷款还没批下来，银行说要等交房。", "simple", "record_only"),
    ("lawyer", "开发商有没有给出新的交房时间？", "ignore", "none"),
    ("client", "说年底能交，但没有书面承诺。", "simple", "record_only"),
    ("client", "逾期交房违约金怎么算？", "simple", "compute_compensation"),
    ("lawyer", "按合同约定，已付房款乘以万分之三再乘以逾期天数。", "ignore", "none"),
    ("client", "我能解除合同吗？", "complex", "query_law"),
    ("lawyer", "逾期超过一定期限可以解除，各地标准不同，一般是90天到180天。", "ignore", "none"),
    ("client", "解除合同能拿到多少赔偿？", "simple", "compute_compensation"),
    ("lawyer", "已付房款加利息加违约金，具体数额要算一下。", "ignore", "none"),
    ("client", "我该起诉还是继续等？", "complex", "strategy_advice"),
    ("lawyer", "建议先发律师函催告，给30天期限，过了还不交再起诉。", "ignore", "none"),
    ("client", "起诉胜率高吗？", "complex", "risk_evaluation"),
    ("lawyer", "合同明确的话胜率很高，关键是执行，开发商可能没钱。", "ignore", "none"),
    ("client", "其他业主也想一起告，能集体诉讼吗？", "complex", "query_law"),
    ("lawyer", "可以推选代表人诉讼，但每个人情况不同，可能要分别立案。", "ignore", "none"),
    ("client", "谢谢律师，我先回去联系其他业主。", "ignore", "none"),

    # ====== 交通事故剧本样本 ======
    ("lawyer", "您好，请问是什么事故？", "ignore", "none"),
    ("client", "律师好，我被车撞了，对方全责。", "simple", "record_only"),
    ("lawyer", "什么时候的事，在哪里？", "ignore", "none"),
    ("client", "上周三在中山路十字路口，对方闯红灯。", "simple", "record_only"),
    ("lawyer", "报警了吗，交警怎么定责？", "ignore", "none"),
    ("client", "报了，交警认定对方全责，有事故认定书。", "simple", "record_only"),
    ("lawyer", "受伤情况怎么样？", "ignore", "none"),
    ("client", "右腿骨折，做了手术，现在还在住院。", "simple", "record_only"),
    ("lawyer", "医疗费花了多少？", "ignore", "none"),
    ("client", "目前 5 万多，后续还要康复。", "simple", "record_only"),
    ("lawyer", "对方有保险吗？", "ignore", "none"),
    ("client", "有交强险和商业险，但保险公司说只赔医保范围内。", "simple", "record_only"),
    ("lawyer", "您的工作和收入情况？", "ignore", "none"),
    ("client", "我是程序员，月薪 1 万 5，现在没法上班。", "simple", "record_only"),
    ("client", "交通事故赔偿包括哪些项目？", "simple", "query_law"),
    ("lawyer", "医疗费、误工费、护理费、营养费、交通费、残疾赔偿金等。", "ignore", "none"),
    ("client", "误工费怎么算？", "simple", "compute_compensation"),
    ("lawyer", "有固定收入的按实际减少算，没固定收入的按平均工资算。", "ignore", "none"),
    ("client", "残疾赔偿金怎么算？", "simple", "compute_compensation"),
    ("lawyer", "按伤残等级、当地人均可支配收入、赔偿年限计算。", "ignore", "none"),
    ("client", "保险公司不赔非医保用药怎么办？", "complex", "strategy_advice"),
    ("lawyer", "可以起诉要求全额赔偿，法院一般支持合理医疗费用。", "ignore", "none"),
    ("client", "大概能赔多少？", "complex", "risk_evaluation"),
    ("lawyer", "看伤残等级，十级的话大概十几万，等级越高越多。", "ignore", "none"),
    ("client", "谢谢律师，我先把材料整理好。", "ignore", "none"),

    # ====== 试用期辞退剧本样本 ======
    ("lawyer", "您好，请问是什么情况？", "ignore", "none"),
    ("client", "律师好，我试用期被公司辞退了。", "simple", "record_only"),
    ("lawyer", "入职多久了？", "ignore", "none"),
    ("client", "两个月，合同签的三年，试用期6个月。", "simple", "record_only"),
    ("lawyer", "公司给出的辞退理由是什么？", "ignore", "none"),
    ("client", "说不符合录用条件，但我没签过录用条件确认书。", "simple", "record_only"),
    ("lawyer", "有没有书面辞退通知？", "ignore", "none"),
    ("client", "有，写的是不符合录用条件，但没有具体说明。", "simple", "record_only"),
    ("lawyer", "工资发到什么时候？", "ignore", "none"),
    ("client", "说当天结算，但只给了半个月工资。", "simple", "record_only"),
    ("client", "试用期被辞退有赔偿吗？", "simple", "query_law"),
    ("lawyer", "违法辞退有赔偿，但如果确实不符合录用条件且程序合法就没有。", "ignore", "none"),
    ("client", "我能拿多少赔偿？", "simple", "compute_compensation"),
    ("lawyer", "如果违法解除，不满半年的按半个月工资两倍，也就是一个月工资。", "ignore", "none"),
    ("client", "我该仲裁还是协商？", "complex", "strategy_advice"),
    ("lawyer", "先协商，协商不成再仲裁，仲裁是免费的。", "ignore", "none"),
    ("client", "仲裁胜率高吗？", "complex", "risk_evaluation"),
    ("lawyer", "没签录用条件确认书的话胜率挺高的。", "ignore", "none"),
    ("client", "谢谢律师，我先去准备材料。", "ignore", "none"),

    # ====== 民间借贷剧本样本 ======
    ("lawyer", "您好，请问是什么纠纷？", "ignore", "none"),
    ("client", "律师好，朋友借了我 10 万不还。", "simple", "record_only"),
    ("lawyer", "什么时候借的，有借条吗？", "ignore", "none"),
    ("client", "去年5月借的，有借条，写的月息2分。", "simple", "record_only"),
    ("lawyer", "钱是怎么给的，转账还是现金？", "ignore", "none"),
    ("client", "银行转账，有转账记录。", "simple", "record_only"),
    ("lawyer", "对方还过多少？", "ignore", "none"),
    ("client", "利息给了3个月，后面就没还了。", "simple", "record_only"),
    ("lawyer", "有没有催过，怎么催的？", "ignore", "none"),
    ("client", "微信催过几次，对方说没钱。", "simple", "record_only"),
    ("client", "民间借贷利息最高多少合法？", "simple", "query_law"),
    ("lawyer", "现在LPR的4倍，大概年化14%左右，月息2分超过了。", "ignore", "none"),
    ("client", "超过部分能要回来吗？", "simple", "query_law"),
    ("lawyer", "超过部分法院不支持，但本金和合法利息可以要回。", "ignore", "none"),
    ("client", "利息怎么算？", "simple", "compute_compensation"),
    ("lawyer", "按合法利率算，本金乘年利率乘时间，减去已还部分。", "ignore", "none"),
    ("client", "我该起诉还是再等等？", "complex", "strategy_advice"),
    ("lawyer", "建议先发律师函，同时准备起诉，借条诉讼时效是3年。", "ignore", "none"),
    ("client", "起诉胜率高吗？", "complex", "risk_evaluation"),
    ("lawyer", "有借条有转账记录胜率很高，主要是执行问题。", "ignore", "none"),
    ("client", "谢谢律师，我尽快准备。", "ignore", "none"),
]

print(f"基础种子样本数: {len(SEED_SAMPLES)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3.2 模板扩充数据集至约 1000 条
# ═══════════════════════════════════════════════════════════════

# 定义各领域关键词库
DOMAINS = {
    "labor": {
        "terms": ["劳动合同", "工资", "加班费", "经济补偿", "N+1", "2N", "违法解除", "辞退", "辞职", "年假", "社保", "公积金", "工伤", "竞业限制", "保密协议", "试用期", "不胜任", "调岗", "降薪", "裁员", "失业金"],
        "questions": ["违法解除赔多少", "N+1怎么算", "加班费怎么算", "年假有多少天", "竞业限制最长多久", "试用期最长多久", "经济补偿怎么算", "工伤怎么认定", "失业金怎么领", "社保没交怎么办"],
        "strategy": ["该怎么跟公司谈", "怎么收集证据", "先仲裁还是协商", "怎么写仲裁申请书", "调解好还是仲裁好", "怎么应对公司威胁"],
        "risk": ["能赢吗", "胜率有多大", "大概能赔多少", "仲裁能支持吗"],
        "compute": ["N+1怎么算", "加班费有多少", "赔偿金有多少", "年假工资怎么算", "经济补偿怎么算"],
        "facts": ["在公司工作了{dur}", "月薪{amt}", "说我不{reason}", "他们说我{reason}", "公司说{reason}", "主管说{reason}", "HR说{reason}", "没有书面通知", "只是口头通知", "合同签的{dur}", "加班{dur}没给加班费"],
    },
    "contract": {
        "terms": ["合同", "违约", "违约金", "履行", "解除", "撤销", "无效", "效力", "条款", "定金", "订金", "预付款", "交付", "验收"],
        "questions": ["违约金怎么算", "定金能退吗", "合同无效怎么办", "怎么解除合同", "对方违约怎么办"],
        "strategy": ["该怎么谈判", "怎么发律师函", "先调解还是起诉", "怎么保全财产"],
        "risk": ["能赢吗", "胜率多大", "违约金能支持多少"],
        "compute": ["违约金怎么算", "定金双倍是多少", "损失有多少"],
        "facts": ["对方没按合同交货", "他们说{reason}", "合同约定{dur}交付", "付了{amt}定金"],
    },
    "criminal": {
        "terms": ["盗窃", "诈骗", "故意伤害", "交通肇事", "醉驾", "寻衅滋事", "职务侵占", "挪用资金", "敲诈勒索", "抢劫", "抢夺"],
        "questions": ["怎么量刑", "能取保吗", "缓刑条件是什么", "认罪认罚能减多少"],
        "strategy": ["该请律师还是法律援助", "怎么争取缓刑", "退赃能减多少", "怎么申请取保"],
        "risk": ["大概判多久", "能缓刑吗", "胜率有多大", "最重判多少"],
        "compute": ["退赃能减多少", "罚金有多少", "刑期怎么算"],
        "facts": ["警察说{reason}", "涉嫌{crime}", "涉案金额{amt}", "之前有过前科", "他说{reason}", "已经{dur}了"],
    },
    "marriage": {
        "terms": ["离婚", "抚养权", "抚养费", "财产分割", "婚前财产", "婚后财产", "共同财产", "过错", "家暴", "出轨"],
        "questions": ["抚养权怎么判", "抚养费怎么算", "婚前财产能分吗", "出轨能要求赔偿吗"],
        "strategy": ["怎么争取抚养权", "怎么查清财产", "先协议还是起诉", "怎么收集出轨证据"],
        "risk": ["抚养权能拿到吗", "大概能分多少", "胜率有多大"],
        "compute": ["抚养费怎么算", "财产分割能拿多少", "精神损害赔偿有多少"],
        "facts": ["结婚{dur}", "孩子{age}岁", "他说{reason}", "我发现他{reason}", "有{amt}存款"],
    },
    "property": {
        "terms": ["房产", "房屋", "开发商", "交房", "逾期", "房产证", "物业", "面积", "公摊", "装修"],
        "questions": ["逾期交房能解除吗", "面积差怎么处理", "公摊面积怎么算"],
        "strategy": ["该起诉还是协商", "怎么集体维权", "怎么发律师函"],
        "risk": ["胜率高吗", "能拿到多少赔偿", "开发商没钱怎么办"],
        "compute": ["违约金怎么算", "面积差价有多少", "逾期赔偿有多少"],
        "facts": ["买房{dur}了", "合同约定{dur}交房", "开发商说{reason}", "付了{amt}首付"],
    },
    "loan": {
        "terms": ["借款", "借条", "欠条", "利息", "本金", "还款", "逾期", "担保", "抵押", "质押"],
        "questions": ["利息多少合法", "没借条能起诉吗", "担保人要承担什么责任"],
        "strategy": ["该怎么催款", "先起诉还是调解", "怎么保全财产"],
        "risk": ["能要回来吗", "胜率有多大", "执行难怎么办"],
        "compute": ["利息怎么算", "逾期利息有多少", "违约金怎么算"],
        "facts": ["借了{amt}", "他说{reason}", "借条写了{dur}", "还了{amt}", "利息{rate}", "说{dur}还"],
    },
    "traffic": {
        "terms": ["交通事故", "责任认定", "全责", "主责", "赔偿", "医疗费", "误工费", "护理费", "营养费", "残疾赔偿金"],
        "questions": ["赔偿包括哪些项目", "误工费怎么算", "残疾赔偿金怎么算"],
        "strategy": ["怎么跟保险公司谈", "保险公司不赔怎么办", "怎么起诉"],
        "risk": ["能赔多少", "伤残等级怎么定", "胜率有多大"],
        "compute": ["误工费怎么算", "残疾赔偿金有多少", "护理费怎么算"],
        "facts": ["对方{reason}", "交警认定{reason}", "住院{dur}", "花了{amt}医疗费", "医生说{reason}", "他们说{reason}"],
    },
}

FILLERS = {
    "dur": ["两年", "三年", "五年", "半年", "一个月", "三个月", "六个月", "一年", "十天", "一周"],
    "amt": ["两万", "五万", "十万", "二十万", "五十万", "一百万", "三千", "五千", "一万"],
    "reason": ["没钱", "不是故意的", "不了解情况", "搞错了", "不认可", "不同意", "没说过", "不知道"],
    "crime": ["盗窃", "诈骗", "故意伤害", "交通肇事"],
    "age": ["3", "5", "8", "10", "12", "1", "2"],
    "rate": ["月息一分", "月息两分", "年息15%", "年息20%"],
}

def fill_template(template):
    """填充模板中的占位符"""
    import re
    result = template
    for key, values in FILLERS.items():
        while f"{{{key}}}" in result:
            result = result.replace(f"{{{key}}}", random.choice(values), 1)
    return result

# 生成扩充样本
generated = []

# 1. ignore/none — 律师询问（大量）
lawyer_asks = [
    "请问您{topic}是什么情况？", "关于{topic}能详细说一下吗？", "{topic}是什么时候的事？",
    "{topic}具体金额是多少？", "您{topic}的证据有哪些？", "{topic}对方怎么说？",
    "之前有没有{topic}的记录？", "{topic}现在到什么阶段了？", "您对{topic}有什么诉求？",
    "除了{topic}还有其他问题吗？", "{topic}有没有书面材料？", "您{topic}多久了？",
    "对方{topic}的理由是什么？", "{topic}造成的损失有多大？", "您{topic}的收入情况？",
    "家里{topic}的情况？", "{topic}的合同约定是什么？", "有没有{topic}的聊天记录？"
]
for domain, data in DOMAINS.items():
    for term in data["terms"]:
        for template in lawyer_asks:
            text = template.format(topic=term)
            generated.append(("lawyer", text, "ignore", "none"))

# 2. ignore/none — 律师解释/陈述
lawyer_explains = [
    "根据{topic}的规定，{desc}", "一般来说{topic}是{desc}", "这个要看{topic}的具体情况",
    "{topic}需要{desc}", "关于{topic}，实务中通常{desc}", "{topic}原则上{desc}",
    "您说的{topic}涉及{desc}", "{topic}的规定比较复杂", "建议先准备{topic}的材料"
]
explain_descs = ["这样处理", "由法院认定", "按合同约定", "看证据情况", "根据法律规定"]
for domain, data in DOMAINS.items():
    for term in data["terms"]:
        for template in lawyer_explains:
            text = template.format(topic=term, desc=random.choice(explain_descs))
            generated.append(("lawyer", text, "ignore", "none"))

# 3. ignore/none — 客户寒暄/应答
client_greetings = [
    "好的", "明白了", "谢谢律师", "嗯嗯", "知道了", "我清楚了",
    "麻烦您了", "好的我记一下", "是这样", "对", "没错",
    "好的律师", "谢谢", "了解了", "我回去准备", "我再想想"
]
for text in client_greetings:
    generated.append(("client", text, "ignore", "none"))

# 4. simple/record_only — 客户陈述事实（大量）
for domain, data in DOMAINS.items():
    for template in data["facts"]:
        text = fill_template(template)
        generated.append(("client", text, "simple", "record_only"))
    # 增加一些通用事实陈述
    extra_facts = [
        "我是{dur}入职的", "月薪大概{amt}", "已经工作{dur}了",
        "他们说{reason}", "对方说{reason}", "公司认为{reason}",
        "主管告诉我{reason}", "HR说{reason}", "对方律师说{reason}"
    ]
    for template in extra_facts:
        text = fill_template(template)
        generated.append(("client", text, "simple", "record_only"))

# 5. simple/query_law — 法条查询
query_templates = [
    "{q}？", "{q}怎么规定？", "{q}是怎么说的？",
    "请问{q}？", "我想了解一下{q}", "{q}有什么规定？"
]
for domain, data in DOMAINS.items():
    for q in data["questions"]:
        for template in query_templates:
            text = template.format(q=q)
            generated.append(("client", text, "simple", "query_law"))

# 6. simple/compute_compensation — 计算
compute_templates = [
    "{q}？", "{q}有多少？", "{q}，您能算一下吗？",
    "请问{q}？", "帮我算一下{q}"
]
for domain, data in DOMAINS.items():
    for q in data.get("compute", data["questions"]):
        for template in compute_templates:
            text = template.format(q=q)
            generated.append(("client", text, "simple", "compute_compensation"))

# 7. complex/strategy_advice — 策略
strategy_templates = [
    "{q}？", "请问{q}？", "律师，{q}？",
    "我想问问{q}", "{q}比较好？"
]
for domain, data in DOMAINS.items():
    for q in data["strategy"]:
        for template in strategy_templates:
            text = template.format(q=q)
            generated.append(("client", text, "complex", "strategy_advice"))

# 8. complex/risk_evaluation — 胜率
risk_templates = [
    "{q}？", "请问{q}？", "律师，{q}？",
    "您觉得{q}？", "我想知道{q}"
]
for domain, data in DOMAINS.items():
    for q in data["risk"]:
        for template in risk_templates:
            text = template.format(q=q)
            generated.append(("client", text, "complex", "risk_evaluation"))

# 9. simple/draft_clause — 起草条款（较少）
draft_templates = [
    "帮我写一份{doc}", "怎么写{doc}", "请帮我起草{doc}",
    "{doc}怎么写比较好", "能帮我看看{doc}吗"
]
docs = ["劳动合同", "借条", "欠条", "还款协议", "离婚协议", "保密协议", "竞业限制协议"]
for doc in docs:
    for template in draft_templates:
        text = template.format(doc=doc)
        generated.append(("client", text, "simple", "draft_clause"))

# 10. complex/summarize — 总结（较少）
summarize_templates = [
    "帮我总结一下目前的情况", "目前有哪些证据", "帮我梳理一下案情",
    "总结一下现有的材料", "目前对我有利的有哪些", "帮我整理一下思路"
]
for template in summarize_templates:
    generated.append(("client", template, "complex", "summarize"))

# 合并种子和生成样本
all_samples = SEED_SAMPLES + generated

# 去重
seen = set()
unique_samples = []
for s in all_samples:
    key = (s[0], s[1], s[2], s[3])
    if key not in seen:
        seen.add(key)
        unique_samples.append(s)

# 如果超过 1000 条，随机采样到 1000；不足则保留全部
if len(unique_samples) > 1000:
    random.shuffle(unique_samples)
    unique_samples = unique_samples[:1000]

print(f"去重后总样本数: {len(unique_samples)}")

# 统计分布
severity_counts = Counter([s[2] for s in unique_samples])
intent_counts = Counter([s[3] for s in unique_samples])
print("\nseverity 分布:")
for k, v in severity_counts.most_common():
    print(f"  {k}: {v} ({v/len(unique_samples)*100:.1f}%)")
print("\nintent_type 分布:")
for k, v in intent_counts.most_common():
    print(f"  {k}: {v} ({v/len(unique_samples)*100:.1f}%)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3.3 数据集划分和 DataLoader
# ═══════════════════════════════════════════════════════════════

# 标签映射
SEVERITY_LABELS = ["ignore", "simple", "complex"]
INTENT_LABELS = [
    "query_law", "compute_compensation", "draft_clause", "summarize",
    "record_only", "strategy_advice", "risk_evaluation", "none"
]

severity2id = {label: i for i, label in enumerate(SEVERITY_LABELS)}
intent2id = {label: i for i, label in enumerate(INTENT_LABELS)}

# 输入格式: "speaker: text"
def format_input(speaker, text):
    return f"{speaker}: {text}"

# 构建数据集
texts = [format_input(s[0], s[1]) for s in unique_samples]
severity_labels = [severity2id[s[2]] for s in unique_samples]
intent_labels = [intent2id[s[3]] for s in unique_samples]

# 划分训练/验证/测试集 (70/15/15)
# 自定义分层：确保每个 intent_type 在三个集合中都有样本
def stratified_split_3way(texts, sev_labels, int_labels, test_ratio=0.15, val_ratio=0.15, seed=42):
    """按 intent_type 分层，保证每类在 val/test 中至少 min_n 条"""
    from collections import defaultdict
    rng = random.Random(seed)
    groups = defaultdict(list)
    for i in range(len(texts)):
        groups[int_labels[i]].append(i)

    train_idx, val_idx, test_idx = [], [], []
    for int_id, idxs in groups.items():
        rng.shuffle(idxs)
        n = len(idxs)
        n_test = max(2, int(round(n * test_ratio)))   # 至少 2 条
        n_val = max(2, int(round(n * val_ratio)))     # 至少 2 条
        if n_test + n_val >= n:                       # 样本极少时保护 train
            n_test = max(1, min(n - 2, n_test))
            n_val = max(1, min(n - 1 - n_test, n_val))
        test_idx.extend(idxs[:n_test])
        val_idx.extend(idxs[n_test:n_test + n_val])
        train_idx.extend(idxs[n_test + n_val:])
    return train_idx, val_idx, test_idx

all_idx = list(range(len(texts)))
train_idx, val_idx, test_idx = stratified_split_3way(texts, severity_labels, intent_labels)

train_texts = [texts[i] for i in train_idx]
train_sev = [severity_labels[i] for i in train_idx]
train_int = [intent_labels[i] for i in train_idx]

val_texts = [texts[i] for i in val_idx]
val_sev = [severity_labels[i] for i in val_idx]
val_int = [intent_labels[i] for i in val_idx]

test_texts = [texts[i] for i in test_idx]
test_sev = [severity_labels[i] for i in test_idx]
test_int = [intent_labels[i] for i in test_idx]

print(f"训练集: {len(train_texts)} 条")
print(f"验证集: {len(val_texts)} 条")
print(f"测试集: {len(test_texts)} 条")
print()
print("训练集 intent 分布:", dict(sorted(Counter(train_int).items(), key=lambda x: x[0])))
print("验证集 intent 分布:", dict(sorted(Counter(val_int).items(), key=lambda x: x[0])))
print("测试集 intent 分布:", dict(sorted(Counter(test_int).items(), key=lambda x: x[0])))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3.4 Tokenizer 和 PyTorch Dataset
# ═══════════════════════════════════════════════════════════════

MODEL_NAME = "hfl/chinese-macbert-base"
MAX_LENGTH = 128

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class IntentDataset(Dataset):
    def __init__(self, texts, severity_labels, intent_labels, tokenizer, max_length):
        self.texts = texts
        self.severity_labels = severity_labels
        self.intent_labels = intent_labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "severity_label": torch.tensor(self.severity_labels[idx], dtype=torch.long),
            "intent_label": torch.tensor(self.intent_labels[idx], dtype=torch.long),
        }

train_dataset = IntentDataset(train_texts, train_sev, train_int, tokenizer, MAX_LENGTH)
val_dataset = IntentDataset(val_texts, val_sev, val_int, tokenizer, MAX_LENGTH)
test_dataset = IntentDataset(test_texts, test_sev, test_int, tokenizer, MAX_LENGTH)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# 查看一个批次
batch = next(iter(train_loader))
print("批次 shape:")
print(f"  input_ids: {batch['input_ids'].shape}")
print(f"  severity_label: {batch['severity_label'].shape}")
print(f"  intent_label: {batch['intent_label'].shape}")

## 第四步：模型定义

使用 MacBERT-base 作为 encoder，两个独立的分类头：
- `severity_classifier` → 3 类（ignore / simple / complex）
- `intent_classifier` → 8 类（query_law / compute_compensation / draft_clause / summarize / record_only / strategy_advice / risk_evaluation / none）

多任务学习：联合优化两个交叉熵损失。

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4.1 模型定义
# ═══════════════════════════════════════════════════════════════

class IntentRouterBERT(nn.Module):
    def __init__(self, model_name, num_severity, num_intent, dropout_rate=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 768

        self.dropout = nn.Dropout(dropout_rate)
        self.severity_classifier = nn.Linear(hidden_size, num_severity)
        self.intent_classifier = nn.Linear(hidden_size, num_intent)

        # 初始化分类头
        nn.init.xavier_uniform_(self.severity_classifier.weight)
        nn.init.zeros_(self.severity_classifier.bias)
        nn.init.xavier_uniform_(self.intent_classifier.weight)
        nn.init.zeros_(self.intent_classifier.bias)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # 取 [CLS] token 的表示
        pooled = outputs.last_hidden_state[:, 0, :]  # [batch, hidden]
        pooled = self.dropout(pooled)

        severity_logits = self.severity_classifier(pooled)   # [batch, 3]
        intent_logits = self.intent_classifier(pooled)       # [batch, 8]

        return severity_logits, intent_logits

model = IntentRouterBERT(MODEL_NAME, len(SEVERITY_LABELS), len(INTENT_LABELS))
model = model.to(DEVICE)

# 打印参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")
print(f"模型大小: {total_params * 4 / 1024 / 1024:.1f} MB (fp32)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4.2 训练配置
# ═══════════════════════════════════════════════════════════════

EPOCHS = 15
LR = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEV_LOSS_WEIGHT = 1.0
INTENT_LOSS_WEIGHT = 1.0

# 优化器
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# 学习率调度
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

# 类别权重（按训练集样本数反比计算，平滑处理）
from collections import Counter

train_sev_counts = Counter(train_sev)
train_int_counts = Counter(train_int)

# severity 权重: 样本越少权重越高，用 sqrt 平滑避免极端权重
sev_weights = torch.tensor([
    1.0 / (train_sev_counts.get(i, 1) ** 0.5) for i in range(len(SEVERITY_LABELS))
], dtype=torch.float)
sev_weights = sev_weights / sev_weights.mean()  # 归一化到均值 1

intent_weights = torch.tensor([
    1.0 / (train_int_counts.get(i, 1) ** 0.5) for i in range(len(INTENT_LABELS))
], dtype=torch.float)
intent_weights = intent_weights / intent_weights.mean()

sev_weights = sev_weights.to(DEVICE)
intent_weights = intent_weights.to(DEVICE)

# 损失函数（带类别权重）
criterion_sev = nn.CrossEntropyLoss(weight=sev_weights)
criterion_intent = nn.CrossEntropyLoss(weight=intent_weights)

print(f"总训练步数: {total_steps}")
print(f"Warmup 步数: {warmup_steps}")
print()
print("Severity 类别权重:", {SEVERITY_LABELS[i]: round(float(sev_weights[i]), 2) for i in range(len(SEVERITY_LABELS))})
print("Intent 类别权重:", {INTENT_LABELS[i]: round(float(intent_weights[i]), 2) for i in range(len(INTENT_LABELS))})

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4.3 训练循环
# ═══════════════════════════════════════════════════════════════

def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    sev_correct = 0
    intent_correct = 0
    total = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        sev_labels = batch["severity_label"].to(device)
        intent_labels = batch["intent_label"].to(device)

        optimizer.zero_grad()
        sev_logits, intent_logits = model(input_ids, attention_mask)

        loss_sev = criterion_sev(sev_logits, sev_labels)
        loss_intent = criterion_intent(intent_logits, intent_labels)
        loss = SEV_LOSS_WEIGHT * loss_sev + INTENT_LOSS_WEIGHT * loss_intent

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        sev_correct += (sev_logits.argmax(dim=1) == sev_labels).sum().item()
        intent_correct += (intent_logits.argmax(dim=1) == intent_labels).sum().item()
        total += sev_labels.size(0)

    return total_loss / len(dataloader), sev_correct / total, intent_correct / total


def eval_epoch(model, dataloader, device):
    model.eval()
    total_loss = 0
    sev_correct = 0
    intent_correct = 0
    both_correct = 0
    total = 0

    all_sev_preds, all_sev_labels = [], []
    all_intent_preds, all_intent_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            sev_labels = batch["severity_label"].to(device)
            intent_labels = batch["intent_label"].to(device)

            sev_logits, intent_logits = model(input_ids, attention_mask)

            loss_sev = criterion_sev(sev_logits, sev_labels)
            loss_intent = criterion_intent(intent_logits, intent_labels)
            loss = SEV_LOSS_WEIGHT * loss_sev + INTENT_LOSS_WEIGHT * loss_intent

            total_loss += loss.item()

            sev_pred = sev_logits.argmax(dim=1)
            intent_pred = intent_logits.argmax(dim=1)

            sev_correct += (sev_pred == sev_labels).sum().item()
            intent_correct += (intent_pred == intent_labels).sum().item()
            both_correct += ((sev_pred == sev_labels) & (intent_pred == intent_labels)).sum().item()
            total += sev_labels.size(0)

            all_sev_preds.extend(sev_pred.cpu().numpy())
            all_sev_labels.extend(sev_labels.cpu().numpy())
            all_intent_preds.extend(intent_pred.cpu().numpy())
            all_intent_labels.extend(intent_labels.cpu().numpy())

    metrics = {
        "loss": total_loss / len(dataloader),
        "sev_acc": sev_correct / total,
        "intent_acc": intent_correct / total,
        "both_acc": both_correct / total,
        "sev_f1": f1_score(all_sev_labels, all_sev_preds, average="weighted", zero_division=0),
        "intent_f1": f1_score(all_intent_labels, all_intent_preds, average="weighted", zero_division=0),
    }
    return metrics


# 训练主循环
best_both_acc = 0
history = {"train_loss": [], "val_loss": [], "val_sev_acc": [], "val_intent_acc": [], "val_both_acc": []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_sev_acc, train_intent_acc = train_epoch(
        model, train_loader, optimizer, scheduler, DEVICE
    )
    val_metrics = eval_epoch(model, val_loader, DEVICE)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_sev_acc"].append(val_metrics["sev_acc"])
    history["val_intent_acc"].append(val_metrics["intent_acc"])
    history["val_both_acc"].append(val_metrics["both_acc"])

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train_loss={train_loss:.4f} sev_acc={train_sev_acc:.3f} intent_acc={train_intent_acc:.3f} | "
          f"val_loss={val_metrics['loss']:.4f} sev_acc={val_metrics['sev_acc']:.3f} "
          f"intent_acc={val_metrics['intent_acc']:.3f} both_acc={val_metrics['both_acc']:.3f}")

    if val_metrics["both_acc"] > best_both_acc:
        best_both_acc = val_metrics["both_acc"]
        # 保存最佳模型
        os.makedirs("checkpoints", exist_ok=True)
        torch.save(model.state_dict(), "checkpoints/best_model.pt")
        print(f"  >>> 保存最佳模型 (both_acc={best_both_acc:.3f})")

print(f"\n训练完成。最佳 both_acc: {best_both_acc:.3f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 第五步：训练过程可视化
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss 曲线
axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"], label="Val Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy 曲线
axes[1].plot(history["val_sev_acc"], label="Severity Acc")
axes[1].plot(history["val_intent_acc"], label="Intent Acc")
axes[1].plot(history["val_both_acc"], label="Both Correct Acc")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 第六步：测试集评估
# ═══════════════════════════════════════════════════════════════

# 加载最佳模型
model.load_state_dict(torch.load("checkpoints/best_model.pt"))
model.eval()

test_metrics = eval_epoch(model, test_loader, DEVICE)
print("测试集指标:")
print(f"  Loss: {test_metrics['loss']:.4f}")
print(f"  Severity Accuracy: {test_metrics['sev_acc']:.3f}")
print(f"  Severity F1: {test_metrics['sev_f1']:.3f}")
print(f"  Intent Accuracy: {test_metrics['intent_acc']:.3f}")
print(f"  Intent F1: {test_metrics['intent_f1']:.3f}")
print(f"  Both Correct: {test_metrics['both_acc']:.3f}")

# 收集测试集预测
all_sev_preds, all_sev_labels = [], []
all_intent_preds, all_intent_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        sev_logits, intent_logits = model(input_ids, attention_mask)
        all_sev_preds.extend(sev_logits.argmax(dim=1).cpu().numpy())
        all_sev_labels.extend(batch["severity_label"].numpy())
        all_intent_preds.extend(intent_logits.argmax(dim=1).cpu().numpy())
        all_intent_labels.extend(batch["intent_label"].numpy())

# 分类报告
print("\n===== Severity 分类报告 =====")
print(classification_report(
    all_sev_labels, all_sev_preds, target_names=SEVERITY_LABELS, digits=3, zero_division=0
))

print("===== Intent Type 分类报告 =====")
print(classification_report(
    all_intent_labels, all_intent_preds,
    labels=list(range(len(INTENT_LABELS))),
    target_names=INTENT_LABELS, digits=3, zero_division=0
))

# ── Badcase 分析 ──
print()
print("===== Badcase 分析（两个都错 或 单个错） =====")
badcases = []
for i in range(len(all_sev_labels)):
    sev_ok = all_sev_labels[i] == all_sev_preds[i]
    int_ok = all_intent_labels[i] == all_intent_preds[i]
    if not sev_ok or not int_ok:
        badcases.append({
            'text': test_texts[i],
            'true_sev': SEVERITY_LABELS[all_sev_labels[i]],
            'pred_sev': SEVERITY_LABELS[all_sev_preds[i]],
            'true_int': INTENT_LABELS[all_intent_labels[i]],
            'pred_int': INTENT_LABELS[all_intent_preds[i]],
        })

for bc in badcases[:20]:  # 最多显示 20 条
    marker = "❌" if (bc['true_sev'] != bc['pred_sev'] and bc['true_int'] != bc['pred_int']) else "⚠️"
    print(f"{marker} [{bc['text']}")
    print(f"   真实: {bc['true_sev']}/{bc['true_int']} → 预测: {bc['pred_sev']}/{bc['pred_int']}")
    print()
print(f"共 {len(badcases)} 条 badcase / {len(test_texts)} 条测试集")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 6.1 混淆矩阵可视化
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Severity 混淆矩阵
cm_sev = confusion_matrix(all_sev_labels, all_sev_preds)
sns.heatmap(cm_sev, annot=True, fmt="d", cmap="Blues",
            xticklabels=SEVERITY_LABELS, yticklabels=SEVERITY_LABELS, ax=axes[0])
axes[0].set_title("Severity Confusion Matrix")
axes[0].set_ylabel("True")
axes[0].set_xlabel("Predicted")

# Intent 混淆矩阵
cm_intent = confusion_matrix(all_intent_labels, all_intent_preds)
sns.heatmap(cm_intent, annot=True, fmt="d", cmap="Blues",
            xticklabels=INTENT_LABELS, yticklabels=INTENT_LABELS, ax=axes[1])
axes[1].set_title("Intent Type Confusion Matrix")
axes[1].set_ylabel("True")
axes[1].set_xlabel("Predicted")
plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 第七步：单条推理演示
# ═══════════════════════════════════════════════════════════════

def predict(speaker, text):
    """单条推理"""
    model.eval()
    input_text = format_input(speaker, text)
    encoding = tokenizer(
        input_text, max_length=MAX_LENGTH, padding="max_length",
        truncation=True, return_tensors="pt"
    )
    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        sev_logits, intent_logits = model(input_ids, attention_mask)

    sev_probs = torch.softmax(sev_logits, dim=1).cpu().numpy()[0]
    intent_probs = torch.softmax(intent_logits, dim=1).cpu().numpy()[0]

    sev_pred = SEVERITY_LABELS[sev_logits.argmax(dim=1).item()]
    intent_pred = INTENT_LABELS[intent_logits.argmax(dim=1).item()]

    return {
        "severity": sev_pred,
        "severity_probs": {SEVERITY_LABELS[i]: float(sev_probs[i]) for i in range(len(SEVERITY_LABELS))},
        "intent_type": intent_pred,
        "intent_probs": {INTENT_LABELS[i]: float(intent_probs[i]) for i in range(len(INTENT_LABELS))},
    }

# 测试一些关键样本
demo_samples = [
    ("client", "违法解除赔多少？"),
    ("client", "律师你好"),
    ("client", "我该怎么跟公司谈？"),
    ("lawyer", "你签劳动合同了吗？"),
    ("lawyer", "系统帮我查一下第47条"),
    ("client", "说我不胜任工作。"),
    ("client", "能赢吗？"),
    ("client", "N+1怎么算？"),
    ("client", "帮我写一份劳动合同"),
    ("client", "帮我总结一下目前的情况"),
]

print("单条推理演示:\n")
for speaker, text in demo_samples:
    result = predict(speaker, text)
    print(f"[{speaker}] {text}")
    print(f"  → severity={result['severity']}, intent={result['intent_type']}")
    print(f"    sev_probs: {result['severity_probs']}")
    print()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 第八步：模型导出
# ═══════════════════════════════════════════════════════════════

# 导出为 HuggingFace 格式（便于后续加载）
output_dir = "intent_router_bert"
os.makedirs(output_dir, exist_ok=True)

# 保存 tokenizer
tokenizer.save_pretrained(output_dir)

# 保存模型 state_dict
torch.save(model.state_dict(), os.path.join(output_dir, "pytorch_model.bin"))

# 保存配置文件
config = {
    "model_name": MODEL_NAME,
    "num_severity": len(SEVERITY_LABELS),
    "num_intent": len(INTENT_LABELS),
    "severity_labels": SEVERITY_LABELS,
    "intent_labels": INTENT_LABELS,
    "max_length": MAX_LENGTH,
    "dropout_rate": 0.3,
}
with open(os.path.join(output_dir, "intent_config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

# 保存标签映射
with open(os.path.join(output_dir, "label_map.json"), "w", encoding="utf-8") as f:
    json.dump({
        "severity2id": severity2id,
        "intent2id": intent2id,
        "id2severity": {v: k for k, v in severity2id.items()},
        "id2intent": {v: k for k, v in intent2id.items()},
    }, f, ensure_ascii=False, indent=2)

# 打印目录内容
print(f"模型已导出到 ./{output_dir}/")
for f in os.listdir(output_dir):
    path = os.path.join(output_dir, f)
    size = os.path.getsize(path)
    print(f"  {f}: {size / 1024 / 1024:.1f} MB")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.1 导出 ONNX（可选，用于推理加速）
# ═══════════════════════════════════════════════════════════════

try:
    !pip install -q onnx onnxruntime
    import onnx
    import onnxruntime as ort

    # 导出 ONNX
    dummy_input_ids = torch.randint(0, tokenizer.vocab_size, (1, MAX_LENGTH)).to(DEVICE)
    dummy_attention_mask = torch.ones((1, MAX_LENGTH), dtype=torch.long).to(DEVICE)

    torch.onnx.export(
        model,
        (dummy_input_ids, dummy_attention_mask),
        "intent_router_bert/model.onnx",
        input_names=["input_ids", "attention_mask"],
        output_names=["severity_logits", "intent_logits"],
        dynamic_axes={
            "input_ids": {0: "batch", 1: "seq"},
            "attention_mask": {0: "batch", 1: "seq"},
            "severity_logits": {0: "batch"},
            "intent_logits": {0: "batch"},
        },
        opset_version=14,
    )
    print("ONNX 模型导出成功: intent_router_bert/model.onnx")
except Exception as e:
    print(f"ONNX 导出失败（可选步骤）: {e}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.2 打包下载（Colab 环境）
# ═══════════════════════════════════════════════════════════════

# 打包为 zip
import shutil
shutil.make_archive("intent_router_bert", "zip", "intent_router_bert")
print("打包完成: intent_router_bert.zip")

# 如果是 Colab，提供下载链接
try:
    from google.colab import files
    files.download("intent_router_bert.zip")
    print("已开始下载...")
except ImportError:
    print("非 Colab 环境，跳过自动下载。请手动下载 intent_router_bert.zip")

## 第九步：下游集成示例

以下代码展示如何在项目中加载和使用训练好的模型，替换现有的 LLM-based IntentRouter。

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 9.1 推理封装类（可直接集成到项目）
# ═══════════════════════════════════════════════════════════════

class IntentRouterBERTInference:
    """
    与现有 IntentRouter 接口兼容的 BERT 推理封装。
    使用方式:
      router = IntentRouterBERTInference("intent_router_bert")
      result = router.classify("违法解除赔多少？", speaker="client")
    """
    def __init__(self, model_dir, device=None):
        import json
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # 加载配置
        with open(os.path.join(model_dir, "intent_config.json"), "r", encoding="utf-8") as f:
            self.config = json.load(f)

        with open(os.path.join(model_dir, "label_map.json"), "r", encoding="utf-8") as f:
            label_map = json.load(f)
            self.id2severity = {int(k): v for k, v in label_map["id2severity"].items()}
            self.id2intent = {int(k): v for k, v in label_map["id2intent"].items()}

        self.severity_labels = self.config["severity_labels"]
        self.intent_labels = self.config["intent_labels"]
        self.max_length = self.config["max_length"]

        # 加载 tokenizer 和模型
        self.tokenizer = BertTokenizer.from_pretrained(model_dir)
        self.model = IntentRouterBERT(
            self.config["model_name"],
            len(self.severity_labels),
            len(self.intent_labels),
            dropout_rate=self.config.get("dropout_rate", 0.3)
        )
        state_dict = torch.load(
            os.path.join(model_dir, "pytorch_model.bin"),
            map_location=self.device
        )
        self.model.load_state_dict(state_dict)
        self.model.to(self.device)
        self.model.eval()

    def classify(self, text: str, speaker: str | None = None):
        """
        同步分类接口（与现有 IntentRouter.classify 返回值格式一致）。
        返回 dict，包含 severity, intent_type, severity_probs, intent_probs。
        """
        speaker = speaker or "uncertain"
        input_text = f"{speaker}: {text}"

        encoding = self.tokenizer(
            input_text, max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt"
        )
        input_ids = encoding["input_ids"].to(self.device)
        attention_mask = encoding["attention_mask"].to(self.device)

        with torch.no_grad():
            sev_logits, intent_logits = self.model(input_ids, attention_mask)

        sev_id = sev_logits.argmax(dim=1).item()
        intent_id = intent_logits.argmax(dim=1).item()
        sev_probs = torch.softmax(sev_logits, dim=1).cpu().numpy()[0]
        intent_probs = torch.softmax(intent_logits, dim=1).cpu().numpy()[0]

        return {
            "severity": self.id2severity[sev_id],
            "intent_type": self.id2intent[intent_id],
            "severity_probs": {self.id2severity[i]: float(sev_probs[i]) for i in range(len(self.severity_labels))},
            "intent_probs": {self.id2intent[i]: float(intent_probs[i]) for i in range(len(self.intent_labels))},
            "input_text": input_text,
        }

# 测试集成封装
inference = IntentRouterBERTInference("intent_router_bert")
result = inference.classify("违法解除赔多少？", speaker="client")
print("集成测试:")
print(f"  severity: {result['severity']}")
print(f"  intent_type: {result['intent_type']}")
print(f"  probs: {result['severity_probs']}")

## 附录：与现有系统的对接建议

1. **接口层**：在 `intent_router.py` 中新增 `IntentRouterBERT` 类，保持与现有 `IntentRouter` 相同的 `classify()` 签名
2. **初始化**：项目启动时加载 BERT 模型，避免每次请求重复加载
3. **降级策略**：BERT 输出概率过低（如最高概率 < 0.6）时，fallback 到现有 LLM-based IntentRouter
4. **性能对比**：BERT-base 单条推理约 10-30ms（GPU），远低于 LLM 的 500-2000ms
5. **持续迭代**：收集线上 badcase，定期重新训练